# PlugIR vs. S2 and S3 Fused Top 10 · Hits@10

All three methods use the same 200 dialogue records and 50,000-image index. They stop when the target first ranks in the top 10, or after 10 answers. Round 0 is the search before any questions. A hit counts for that round and every later round.

- **S2 history pool** uses reciprocal rank fusion across all distinct queries so far to fill the candidate pool, then runs PlugIR’s k-means and representative selection as usual.
- **S3 Fused Top 10** keeps the candidate pool from the latest query and sends the captions of its 10 highest history-ranked images to the questioner.

Both variants use the latest query for user-facing results. They keep the same candidate-pool schedule and pinned BLIP and Qwen checkpoints. This notebook reuses the saved vanilla PlugIR records from the baseline and S1 run.

The progress and plotting cells can read the included saved results on a CPU. The run cell requires an Apple Silicon Mac and the two environments described in the README.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# Start Jupyter in the directory containing this notebook, or set PACKAGE explicitly.
PACKAGE = Path.cwd().resolve()
RESULTS = PACKAGE / "results"
REPORT = PACKAGE / "source" / "report_hits.py"

# Change these paths if your checkout or MLX environment lives elsewhere.
PROJECT = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / "ChatIR-Cornell2026").is_dir()),
    PACKAGE.parent,
)
REPO = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / ".git").exists()),
    PROJECT / "ChatIR-Cornell2026",
)
BLIP_PYTHON = REPO / ".venv" / "bin" / "python"
MLX_ENV = PROJECT / "tmp" / "agentir-mlx-vlm-env"
QWEN_REVISION = "8b2b98c00a6b4d291155e4890773ca8f769aee53"
SNAPSHOT = (REPO / "cache/huggingface/hub/models--mlx-community--Qwen3.5-9B-4bit"
            / "snapshots" / QWEN_REVISION)
ARMS = ["s2_history_pool", "s3_fused_top10"]
REPORT_ARMS = ["vanilla", *ARMS]
SAMPLE = json.loads((PACKAGE / "sample.json").read_text())
DIALOGUE_COUNT = len(SAMPLE["tasks"])
print(f"{DIALOGUE_COUNT} dialogues per method, seed {SAMPLE['seed']}")

## Check saved progress

A file is written for every completed dialogue. The matched count includes IDs with a saved result for all three methods.

In [ ]:
def saved_ids(arm):
    return {int(path.stem) for path in (RESULTS / arm / "dialogues").glob("*.json")}

for arm in REPORT_ARMS:
    print(f"{arm}: {len(saved_ids(arm))}/{DIALOGUE_COUNT}")
print(f"Matched: {len(set.intersection(*(saved_ids(arm) for arm in REPORT_ARMS)))}/{DIALOGUE_COUNT}")

## Run or resume on Apple Silicon

Run this cell only when you want to start inference. If a process is already writing to this results folder, use the progress cell. Set `RESULTS` to a new directory before starting an independent replication. The run cell copies the included baseline records to a fresh results directory if needed. The saved settings and source hashes are checked on resume.

In [ ]:
from shutil import copy2, copytree
from tempfile import TemporaryDirectory

for dependency in (BLIP_PYTHON, MLX_ENV / "bin/python", SNAPSHOT / "config.json"):
    if not dependency.exists():
        raise FileNotFoundError(f"{dependency} is missing. See README.md for setup.")

reference = PACKAGE / "results"
if RESULTS.resolve() != reference.resolve():
    RESULTS.mkdir(parents=True, exist_ok=True)
    if not (RESULTS / "vanilla_config.json").exists():
        copytree(reference / "vanilla", RESULTS / "vanilla", dirs_exist_ok=True)
        copy2(reference / "vanilla_config.json", RESULTS / "vanilla_config.json")

# Stage the recorded source in the layout used by the original runner.
with TemporaryDirectory(prefix="agentir-s2-s3-") as stage:
    workspace = Path(stage)
    bundle_dir = workspace / "deliverables/full_dialogue_colab"
    source_dir = bundle_dir / "source"
    source_dir.mkdir(parents=True)
    (workspace / "ChatIR-Cornell2026").symlink_to(REPO, target_is_directory=True)
    (workspace / "tmp").mkdir()
    (workspace / "tmp/agentir-mlx-vlm-env").symlink_to(MLX_ENV, target_is_directory=True)
    for source in (PACKAGE / "source").glob("*.py"):
        copy2(source, source_dir / source.name)
    archive = "agentir_full_dialogue_bundle_v2.zip"
    (bundle_dir / archive).symlink_to(PACKAGE / archive)
    subprocess.run(
        ["caffeinate", "-i", str(BLIP_PYTHON), "-u", str(source_dir / "local_hits.py"),
         "--snapshot", str(SNAPSHOT), "--output", str(RESULTS), "--limit", str(DIALOGUE_COUNT),
         "--arms", *ARMS],
        check=True,
    )

## Plot the matched comparison

The default requires all 200 dialogues across the three methods. Set `ALLOW_PARTIAL = True` to inspect an interim curve labeled with its smaller sample size.

In [ ]:
from IPython.display import Image, display

ALLOW_PARTIAL = False
command = [sys.executable, str(REPORT), str(RESULTS)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")
figures = subprocess.check_output(command, text=True).strip()
display(Image(filename=str(Path(figures) / "hits_at_10.png")))